### 04B · XGBoost del periodo (Tm02)
Misma estructura que el 04 (Hs): 40 PCs (30 presión + 10 gradiente reconstruido, dominio WMed) con desfases 0–36 h → Z de `VTM02` (PIT + armónica del 00_PIT) suavizada 6 h a 00/06/12/18. **Solo PCs, sin la f de Hs**: añadirla daba +0.01 de correlación OOF (0.806 → 0.817); la dependencia Hs–Tm02 que no explica la atmósfera va al ruido conjunto (05B).

Salida: `outputs/xgb_{point}_tm/` (modelo, columnas y residuos horarios). No toca las salidas de Hs (`outputs/xgb_{point}/`).

In [1]:
import os, json, pickle

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold

#### Load and Prepare Data

In [2]:
point = 'p102'

#### Predictores: 30 PCs de presión + 10 PCs del gradiente reconstruido (6 h)
pcs_msl = pd.read_csv('outputs/pca/pcs_msl_30_6h.csv', index_col=0, parse_dates=True)
pcs_grad = pd.read_csv('outputs/grad_rec/pcs_grad_rec_6h.csv', index_col=0, parse_dates=True)
data_pcs = pcs_msl.join(pcs_grad)

#### Objetivo: Z del periodo (PIT + armónica, 00_PIT), media móvil de 6 h centrada, a 00/06/12/18
waves_df_z = pd.read_csv(f'outputs/marginal/z_bulk_{point}.csv', index_col=0, parse_dates=True)
waves_df_z_6h = waves_df_z.rolling('6h', center=True).mean()
waves_df_z_6h = waves_df_z_6h[waves_df_z_6h.index.hour.isin([0, 6, 12, 18])].dropna()

In [3]:
lags = [0, 6, 12, 18, 24, 30, 36]
X_lagged = pd.concat({f'l{l}': data_pcs.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(data_pcs.index)
X_lagged.columns = [f'{c}_l{l[1:]}' for l, c in X_lagged.columns]
X_lagged = X_lagged.dropna()

idx = X_lagged.index.intersection(waves_df_z_6h.index)
X = X_lagged.loc[idx]
y_p = waves_df_z_6h.loc[idx, 'VTM02']
print(X.shape)

(60992, 280)


#### XGBoost (profundidad 3, mismos parámetros que el 04)

In [4]:
xgb_params = dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=200,
                  subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, random_state=0)

#### CrossValidation (1985–2015, pliegues por años)

In [5]:
tr = idx < '2016-01-01'
X_tr, y_p_tr = X[tr], y_p[tr]

f_oof_p = pd.Series(np.nan, index=X_tr.index)
for i_fit, i_val in GroupKFold(n_splits=5).split(X_tr, y_p_tr, groups=X_tr.index.year):
    model = XGBRegressor(**xgb_params).fit(X_tr.iloc[i_fit], y_p_tr.iloc[i_fit])
    f_oof_p.iloc[i_val] = model.predict(X_tr.iloc[i_val])

#### Full model

In [6]:
xgb_tm = XGBRegressor(**xgb_params).fit(X_tr, y_p_tr)
f_test_p = pd.Series(xgb_tm.predict(X[~tr]), index=X.index[~tr])

#### Métricas

In [7]:
print(f"6-horaria OOF:  {f_oof_p.corr(y_p_tr):.3f}")
print(f"6-horaria test: {f_test_p.corr(y_p[~tr]):.3f}")
print(f"diaria OOF:     {f_oof_p.resample('D').mean().corr(y_p_tr.resample('D').mean()):.3f}")
print(f"diaria test:    {f_test_p.resample('D').mean().corr(y_p[~tr].resample('D').mean()):.3f}")

6-horaria OOF:  0.806
6-horaria test: 0.783
diaria OOF:     0.858
diaria test:    0.836


#### Guardar (entrada del 05B)

In [8]:
out_dir = f"outputs/xgb_{point}_tm"
os.makedirs(out_dir, exist_ok=True)

xgb_tm.save_model(f"{out_dir}/xgb_final.json")
with open(f"{out_dir}/xgb_features.json", "w") as f:
    json.dump({"columns": list(X_tr.columns), "lags_h": lags, "params": xgb_params,
               "train": ["1985-01-01", "2015-12-31"], "variable": "VTM02"}, f, indent=1)

#### Serie horaria: z, f y residuo del periodo (OOF en 1985–2015, modelo final en 2016–2025)
f_h = pd.concat([f_oof_p, f_test_p]).sort_index().resample("h").interpolate()
z_h = waves_df_z["VTM02"].reindex(f_h.index)
res_tm = pd.DataFrame({"z": z_h, "f": f_h, "r": z_h - f_h})
res_tm["periodo"] = np.where(res_tm.index < "2016-01-01", "oof", "test")
res_tm.dropna(subset=["r"]).to_csv(f"{out_dir}/residuos_horarios.csv")
print(res_tm.dropna().groupby("periodo")["r"].agg(["count", "std"]))

          count       std
periodo                  
oof      271728  0.615091
test      94219  0.630649
